In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

trips = spark.table("urban_mobility.silver.trips_enriched").filter(F.col("trip_status") == "COMPLETED")
zones = spark.table("urban_mobility.silver.zones")

# Peak hour per route: most frequent pickup_hour for that pickup/dropoff pair
hour_counts = (
    trips.groupBy("pickup_zone_id", "dropoff_zone_id", "pickup_hour")
    .agg(F.count("*").alias("hour_count"))
)
window_spec = Window.partitionBy("pickup_zone_id", "dropoff_zone_id").orderBy(F.desc("hour_count"))
peak_hours = (
    hour_counts
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .select("pickup_zone_id", "dropoff_zone_id", F.col("pickup_hour").alias("peak_hour"))
)

route_agg = (
    trips
    .groupBy("pickup_zone_id", "dropoff_zone_id")
    .agg(
        F.count("trip_id").alias("trip_count"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("distance_km"), 2).alias("avg_distance_km"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("average_speed_kmh"), 2).alias("avg_speed_kmh"),
    )
)

route_performance = (
    route_agg
    .join(peak_hours, ["pickup_zone_id", "dropoff_zone_id"], "left")
    .join(
        zones.select(F.col("zone_id").alias("pickup_zone_id"), F.col("zone_name").alias("pickup_zone_name")),
        "pickup_zone_id", "left"
    )
    .join(
        zones.select(F.col("zone_id").alias("dropoff_zone_id"), F.col("zone_name").alias("dropoff_zone_name")),
        "dropoff_zone_id", "left"
    )
    .select(
        "pickup_zone_id", "pickup_zone_name", "dropoff_zone_id", "dropoff_zone_name",
        "trip_count", "avg_duration_minutes", "avg_distance_km", "avg_fare",
        "total_revenue", "avg_speed_kmh", "peak_hour"
    )
)

route_performance.write.mode("overwrite").format("delta").saveAsTable("urban_mobility.gold.route_performance")

result = spark.table("urban_mobility.gold.route_performance")
print("gold.route_performance rows:", result.count())
result.orderBy(F.desc("trip_count")).show(5)